In [ ]:
"""
===============================================================================
EXERCISE 6.11 (a) - OVERRELAXATION ERROR DERIVATION THEORY
===============================================================================

1. DEFINITION OF THE MODIFIED UPDATING FUNCTION:
------------------------------------------------
In the overrelaxation method, the iteration equation is given by:
    x' = (1 + w)*f(x) - w*x

Let us define this update rule as a new function g(x):
    g(x) = (1 + w)*f(x) - w*x

Since the true solution x* satisfies x* = f(x*):
    g(x*) = (1 + w)*f(x*) - w*x*
          = (1 + w)*x* - w*x*
          = x*
This proves that x* is also the fixed point of g(x).


2. DERIVATION OF THE EQUIVALENT OF EQ. (6.81) (TAYLOR EXPANSION):
-----------------------------------------------------------------
Performing a Taylor series expansion of g(x) around the true solution x*:
    x' = g(x) ≈ g(x*) + (x - x*)*g'(x*)

Since g(x*) = x*:
    x' ≈ x* + (x - x*)*g'(x*)
    x' - x* ≈ (x - x*)*g'(x*)

Defining the error at the current estimate as  e  = x* - x 
and the error at the next estimate as    e' = x* - x':
    -e' ≈ -e * g'(x*)  =>  e' ≈ e * g'(x*)

Taking the derivative of g(x) with respect to x:
    g'(x) = (1 + w)*f'(x) - w

Substituting g'(x*) yields the equivalent of Eq. (6.81) for overrelaxation:
    e' ≈ e * [(1 + w)*f'(x*) - w]   <-- [Eq. 6.81 Equivalent]


3. DERIVATION OF THE EQUIVALENT OF EQ. (6.83) (ERROR ESTIMATION FORMULA):
-------------------------------------------------------------------------
The true solution x* can be expressed in two ways:
    x* = x + e        (In terms of the current estimate)
    x* = x' + e'      (In terms of the next estimate)

Equating both expressions:
    x + e = x' + e'  =>  e = (x' - x) + e'

From e' ≈ e * g'(x*), we isolate e:
    e ≈ e' / g'(x*)

Substitute this e into x + e = x' + e':
    x + [e' / g'(x*)] = x' + e'
    x - x' = e' - [e' / g'(x*)]
    x - x' = e' * [1 - 1/g'(x*)]

Solving for the new error e':
    e' ≈ (x - x') / [1 - 1/g'(x*)]

Since x is close to x*, we approximate g'(x*) ≈ g'(x).
Plugging in g'(x) = (1 + w)*f'(x) - w gives the required error formula:

    e' ≈ (x - x') / [ 1 - 1 / ((1 + w)*f'(x) - w) ]   [Q.E.D.]
===============================================================================
"""

In [1]:
# b

import numpy as np

# step counter 
step = 0

# constants
c = 2
accuracy = 1e-6
x0 = 1
error = 1.0

# calculating the first two
x1 = 1 - np.exp(-c*x0)
x2 = 1 - np.exp(-c*x1)

# iteration loop
while abs(x2-x1) > accuracy:
     # Scroll to the next step
        x0 = x1
        x1 = x2
        x2 = 1 - np.exp(-c*x1) # calculating new guess
        step += 1
error = (x1-x2)**2 / (2*x1-x0-x2)

print("the solution found:",x2)
print("estimated error",abs(error))
print('step number',step)


the solution found: 0.7968126311118457
estimated error 5.010892644903239e-07
step number 12


In [7]:
# c
import numpy as np

# step counter
step = 0

# constants
c = 2
error = 1.0
x0 = 1
accuracy = 1e-6
w = 0.5

# calculating the first two
x1 = (1+w) * (1 - np.exp(-c*x0)) - w * x0
x2 = (1+w) * (1 - np.exp(-c*x1)) - w * x1

# iteration loop
while abs(x2-x1) > accuracy:
    # Scroll to the next step
    x0 = x1
    x1 = x2
    x2 = (1+w) * (1 - np.exp(-c*x1)) - w * x1 # calculating new guess
    step += 1

f_prime = c * np.exp(-c * x1)
g_prime = (1 + w) * f_prime - w
error = (x1 - x2) / (1 - 1 / g_prime)

print("the solution found:",x2)
print("estimated error",abs(error))
print('step number',step)

the solution found: 0.7968121566399141
estimated error 2.661985344348409e-08
step number 3


In [ ]:
"""
===============================================================================
EXERCISE 6.11 (d) - USEFULNESS OF NEGATIVE OMEGA (w < 0 / UNDER-RELAXATION)
===============================================================================

1. DIRECT ANSWER:
-----------------
Yes, using a negative parameter (w < 0), also known as "under-relaxation", 
can significantly speed up convergence—or make a divergent system converge—
when the derivative of the iteration function at the fixed point is negative,
i.e., when f'(x*) < 0.


2. MATHEMATICAL PROOF & REASONING:
----------------------------------
From part (a), the error propagation factor per iteration in overrelaxation is:
    g'(x*) = (1 + w)*f'(x*) - w

The rate of convergence depends entirely on the absolute magnitude |g'(x*)|:
  - If |g'(x*)| is close to 0, convergence is extremely fast.
  - If |g'(x*)| >= 1, the iteration diverges.


3. CASE 1: DAMPING OSCILLATIONS (0 > f'(x*) > -1)
------------------------------------------------
When f'(x*) < 0, the standard relaxation method (w = 0) causes successive 
estimates to oscillate back and forth around the true solution x*.

If we use a positive w (w > 0):
    g'(x*) = (1 + w)*f'(x*) - w
Since both terms are negative, g'(x*) becomes EVEN MORE negative (larger in magnitude).
This amplifies oscillations and SLOWS DOWN convergence.

However, if we choose a negative w (w < 0, let w = -|w|):
    g'(x*) = (1 - |w|)*f'(x*) + |w|
Since f'(x*) < 0 and |w| > 0, the negative term and positive term partially 
cancel each other out, driving g'(x*) closer to 0!

Optimal choice: Set g'(x*) = 0 to get immediate one-step convergence:
    (1 + w)*f'(x*) - w = 0  =>  w = f'(x*) / [1 - f'(x*)]

Since f'(x*) < 0, this optimal w value is strictly NEGATIVE (w < 0).


4. CASE 2: STABILIZING DIVERGENT ITERATIONS (f'(x*) < -1)
--------------------------------------------------------
If f'(x*) < -1, ordinary relaxation (w = 0) completely fails and diverges 
because |f'(x*)| > 1.

By selecting an appropriate negative w (w < 0), we can shrink |g'(x*)| 
so that |g'(x*)| < 1. This stabilizes an otherwise impossible, divergent 
system and allows it to converge rapidly.
===============================================================================
"""